# 🧬 Model 4: Hybrid CNN + LSTM

**Input:** `(batch, N_BANDS, n_channels)`  
**Feature output:** 128-D penultimate vector → SVM

## 0. Mount Google Drive
Required since this notebook reads/writes files under `/content/drive/MyDrive/...`.

FIX: this cell was missing entirely from the original notebook, causing a
FileNotFoundError on dataset_A.csv/dataset_B.csv.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import subprocess
print(subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout or '⚠️ No GPU')

Thu Oct  1 13:20:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!pip install scikit-learn openpyxl -q

In [4]:
import os,gc,time,numpy as np,pandas as pd
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers,models,Input,optimizers as opt_module

BAND_NAMES=["Delta","Theta","Alpha","Beta","Gamma"]

def load_dataset(p):
    # Read the file depending on its format (Excel or CSV)
    df = pd.read_excel(p) if p.endswith(".xlsx") else pd.read_csv(p)
    # Identify columns containing float values and convert them to float32
    # to reduce memory usage
    float_cols = df.select_dtypes(include=["float64"]).columns
    df[float_cols] = df[float_cols].astype(np.float32)
    # NaN guard: a handful of windows can contain NaN feature values.
    n_nan = df.isna().any(axis=1).sum()
    if n_nan:
        print(f"    [nan guard] dropping {n_nan} rows with NaN features")
        df = df.dropna().reset_index(drop=True)
    return df

def normalize_tensors(Xtr, Xv, Xte):
    """Fit mean/std on TRAIN split only (per-feature), apply to all splits."""
    mean = Xtr.mean(axis=0, keepdims=True)
    std  = Xtr.std(axis=0, keepdims=True)
    Xtr  = (Xtr - mean) / (std + 1e-8)
    Xv   = (Xv  - mean) / (std + 1e-8)
    Xte  = (Xte - mean) / (std + 1e-8)
    return Xtr, Xv, Xte
def get_band_columns(df): return {b:[c for c in df.columns if c.startswith(f"{b}_")] for b in BAND_NAMES}
def prepare_tensors_seq(df):
    bc=get_band_columns(df)
    X=np.stack([df[bc[b]].values for b in BAND_NAMES],axis=1)  # (N,5,n_ch)
    return X, df["label"].values.astype(int)
def evaluate_with_svm(Xtr, ytr, Xte, yte, max_train=20000):
    if len(Xtr) > max_train:
        rng = np.random.RandomState(42)
        idx = rng.choice(len(Xtr), max_train, replace=False)
        Xtr, ytr = Xtr[idx], ytr[idx]
    sc = StandardScaler(); Xtr = sc.fit_transform(Xtr); Xte = sc.transform(Xte)
    svm = SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced", random_state=42)
    svm.fit(Xtr, ytr); p = svm.predict(Xte)

    tn, fp, fn, tp = confusion_matrix(yte, p, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0

    return {
        "accuracy": round(accuracy_score(yte, p), 4),
        "sensitivity": round(sensitivity, 4),
        "specificity": round(specificity, 4),
        "f1_macro": round(f1_score(yte, p, average="macro"), 4),
        "f1_weighted": round(f1_score(yte, p, average="weighted"), 4),
        "report": classification_report(yte, p, labels=[0, 1], target_names=["No Seizure", "Seizure"])
    }
def save_results(row,path):
    os.makedirs(os.path.dirname(path),exist_ok=True)
    pd.DataFrame([row]).to_csv(path,mode="a",header=not os.path.exists(path),index=False)
    print("Saved:",row)
def compile_model(m, opt):
    m.compile(optimizer=opt, loss="binary_crossentropy",
              metrics=["accuracy", tf.keras.metrics.AUC(name="auc"),
                       tf.keras.metrics.Precision(name="precision"),
                       tf.keras.metrics.Recall(name="recall")])
    return m


In [5]:
import tensorflow as tf
import numpy as np

# 1. Ensure reproducibility (Crucial for LSTMs)
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

# 2. Prevent GPU Out-Of-Memory (OOM) errors
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

In [6]:
DATASET_A  = "/content/drive/MyDrive/epilepsy_outputs/dataset_A.csv"
DATASET_B  = "/content/drive/MyDrive/epilepsy_outputs/dataset_B.csv"
RESULTS    = "/content/drive/MyDrive/epilepsy_outputs/results.csv"
OUTPUT_DIR = "/content/drive/MyDrive/epilepsy_outputs"
EPOCHS=50; BATCH_SIZE=64; PATIENCE=8; SEED=42
MODEL_NAME = "cnn_lstm"

## Architecture

```
Input -> Reshape(N_BANDS, n_ch, 1)
      -> TimeDistributed(Conv1D(32)) -> TimeDistributed(Conv1D(64))
      -> TimeDistributed(GlobalAvgPool)    # collapse channels per band
      -> BiLSTM(128) -> BN
      -> Dense(128, ReLU) ← feature vector -> Dense(1, sigmoid)
```

In [7]:
from tensorflow.keras import regularizers

def build_cnn_lstm(n_bands=5, n_channels=18, filters=[32, 64], lstm_units=[64], feature_dim=128, dropout=0.3, l2=1e-4, output_bias=None):
    bias_init = tf.keras.initializers.Constant(output_bias) if output_bias is not None else "zeros"
    inp = layers.Input(shape=(n_bands, n_channels), name="eeg_input")

    # Reshape for Conv1D: (batch, time_steps=n_bands, spatial_dims=n_channels, channels=1)
    x = layers.Reshape((n_bands, n_channels, 1))(inp)

    # --- CNN Block ---
    for i, f in enumerate(filters):
        # Change kernel size from 3 to 1 to isolate channels, and add L2
        conv = layers.Conv1D(f, 1, padding="same", activation="relu", kernel_regularizer=regularizers.l2(l2))
        x = layers.TimeDistributed(conv, name=f"td_conv_{i}")(x)
        x = layers.TimeDistributed(layers.BatchNormalization(), name=f"td_bn_{i}")(x)
        # Using spatial dropout inside the spatial dimensions if needed, but since kernel is 1, regular dropout is fine too
        x = layers.TimeDistributed(layers.Dropout(dropout/2), name=f"td_drop_{i}")(x)

    # Collapse spatial dimension -> (batch, n_bands, filters)
    x = layers.TimeDistributed(layers.GlobalAveragePooling1D(), name="td_gap")(x)
    x = layers.Dropout(dropout)(x) # Ordinary dropout is correct here

    # --- LSTM Block ---
    for i, u in enumerate(lstm_units):
        ret = (i < len(lstm_units) - 1)
        # FIX: recurrent_dropout forces TF off the fast cuDNN LSTM kernel onto a
        # much slower unrolled path (~10-15x slowdown). With only 5 timesteps
        # here, its regularization benefit is minimal -- removed for speed.
        lstm = layers.LSTM(u, return_sequences=ret, dropout=dropout,
                           kernel_regularizer=regularizers.l2(l2),
                           recurrent_regularizer=regularizers.l2(l2), name=f"lstm_{i}")
        x = layers.Bidirectional(lstm, name=f"bilstm_{i}")(x)
        x = layers.BatchNormalization(name=f"bn_lstm_{i}")(x)
        if ret:
            x = layers.Dropout(dropout)(x)

    # --- Classification Head ---
    feat = layers.Dense(feature_dim, activation="relu", name="feature_vector", kernel_regularizer=regularizers.l2(l2))(x)
    feat = layers.Dropout(dropout)(feat)
    out = layers.Dense(1, activation="sigmoid", name="output", bias_initializer=bias_init)(feat)

    return models.Model(inp, out, name="CNN_LSTM"), models.Model(inp, feat, name="CNN_LSTM_Feat")

In [8]:
model, feat_ext = build_cnn_lstm()
model.summary()
print("Feature extractor output:", feat_ext.output_shape)

# Clean up the dummy graph before starting the real training loop
del model, feat_ext
tf.keras.backend.clear_session()

Model: "CNN_LSTM"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ eeg_input (InputLayer)          │ (None, 5, 18)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 5, 18, 1)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ td_conv_0 (TimeDistributed)     │ (None, 5, 18, 32)      │            64 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ td_bn_0 (TimeDistributed)       │ (None, 5, 18, 32)      │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ td_drop_0 (TimeDistributed)     │ (None, 5, 18, 32)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ td_conv_1 (TimeDistributed)     │ (None, 5, 18, 64)      │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ td_bn_1 (TimeDistributed)       │ (None, 5, 18, 64)      │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ td_drop_1 (TimeDistributed)     │ (None, 5, 18, 64)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ td_gap (TimeDistributed)        │ (None, 5, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 5, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bilstm_0 (Bidirectional)        │ (None, 128)            │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_lstm_0 (BatchNormalization)  │ (None, 128)            │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ feature_vector (Dense)          │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 85,761 (335.00 KB)

 Trainable params: 85,313 (333.25 KB)

 Non-trainable params: 448 (1.75 KB)

Feature extractor output: (None, 128)


In [9]:
def run_trial(ds_path, ds_label, opt_name, opt_obj):
    print(f"\n{'='*55}\n  {MODEL_NAME} | Dataset {ds_label} | {opt_name}\n{'='*55}")
    df=load_dataset(ds_path); X,y=prepare_tensors_seq(df)
    n_bands,n_ch=X.shape[1],X.shape[2]
    Xtrv,Xte,ytrv,yte=train_test_split(X,y,test_size=0.15,stratify=y,random_state=SEED)
    Xtr,Xv,ytr,yv=train_test_split(Xtrv,ytrv,test_size=0.15/0.85,stratify=ytrv,random_state=SEED)
    Xtr,Xv,Xte=normalize_tensors(Xtr,Xv,Xte)   # train-only normalization
    total=len(ytr); n1=ytr.sum(); cw={0:total/(2*(total-n1)),1:total/(2*n1)}
    output_bias=np.log(n1/(total-n1))          # prior-informed bias init
    ckpt=os.path.join(OUTPUT_DIR,"checkpoints",f"{MODEL_NAME}_{ds_label}_{opt_name}_best.keras")
    epoch_log=ckpt+".epochs.csv"
    os.makedirs(os.path.dirname(ckpt),exist_ok=True)

    # EPOCH-LEVEL RESUME: if a checkpoint + epoch log exist from an interrupted
    # attempt at THIS trial, reload the saved model and continue from where it
    # left off instead of retraining from epoch 0. Note: EarlyStopping/
    # ReduceLROnPlateau's internal "best so far" trackers reset on resume --
    # a minor limitation, but far cheaper than losing all progress on disconnect.
    initial_epoch=0
    if os.path.exists(ckpt) and os.path.exists(epoch_log):
        initial_epoch=len(pd.read_csv(epoch_log))
        print(f"    [resume] found checkpoint at epoch {initial_epoch}, continuing from there")
        full_m=tf.keras.models.load_model(ckpt, compile=False)
        full_m=compile_model(full_m,opt_obj)
        feat_m=models.Model(full_m.input, full_m.get_layer("feature_vector").output)
    else:
        full_m,feat_m=build_cnn_lstm(n_bands=n_bands,n_channels=n_ch,output_bias=output_bias)
        full_m=compile_model(full_m,opt_obj)

    cbs=[tf.keras.callbacks.EarlyStopping(monitor="val_auc",mode="max",patience=PATIENCE,restore_best_weights=True,verbose=1),
         tf.keras.callbacks.ModelCheckpoint(ckpt,monitor="val_auc",mode="max",save_best_only=True,verbose=0),
         tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss",factor=0.5,patience=4,min_lr=1e-6,verbose=1),
         tf.keras.callbacks.CSVLogger(epoch_log, append=True)]
    t0=time.time()
    hist=full_m.fit(Xtr,ytr,validation_data=(Xv,yv),epochs=EPOCHS,initial_epoch=initial_epoch,
                    batch_size=BATCH_SIZE,class_weight=cw,callbacks=cbs,verbose=1)
    elapsed=round((time.time()-t0)/60,2)
    best_auc=max(hist.history.get("val_auc",[0]))
    print(f"  Done {elapsed}min | Best AUC:{best_auc:.4f}")
    f_tr=feat_m.predict(Xtr,batch_size=BATCH_SIZE,verbose=0)
    f_te=feat_m.predict(Xte,batch_size=BATCH_SIZE,verbose=0)
    res=evaluate_with_svm(f_tr,ytr,f_te,yte)
    print(f"  SVM Acc:{res['accuracy']} F1:{res['f1_macro']}\n",res["report"])
    save_results({"model":MODEL_NAME,"dataset":f"Dataset_{ds_label}","optimizer":opt_name,
                  "best_val_auc":round(best_auc,4),**{k:v for k,v in res.items() if k!="report"},
                  "train_time_min":elapsed},RESULTS)
    if os.path.exists(epoch_log):
        os.remove(epoch_log)  # trial finished -- clear resume state
    del full_m,feat_m; gc.collect(); tf.keras.backend.clear_session()

# RESUME SUPPORT: skip any (dataset, optimizer) combo already saved in
# RESULTS, so a runtime disconnect doesn't force a full restart from scratch.
already_done = set()
if os.path.exists(RESULTS):
    _prev = pd.read_csv(RESULTS)
    _prev_this_model = _prev[_prev["model"] == MODEL_NAME]
    already_done = set(zip(_prev_this_model["dataset"], _prev_this_model["optimizer"]))
    if already_done:
        print(f"[resume] found {len(already_done)} completed trial(s) already in {RESULTS}, skipping those:")
        for d, o in sorted(already_done):
            print(f"    - {d} | {o}")

for ds_label,ds_path in [("A",DATASET_A),("B",DATASET_B)]:
    for opt_name,opt_obj in {"Adam":opt_module.Adam(1e-3, clipnorm=1.0),
                             "Nadam":opt_module.Nadam(1e-3, clipnorm=1.0),
                              "AdamW":opt_module.AdamW(1e-3,weight_decay=1e-4, clipnorm=1.0)}.items():
        if (f"Dataset_{ds_label}", opt_name) in already_done:
            print(f"[resume] skipping {MODEL_NAME} | Dataset {ds_label} | {opt_name} (already done)")
            continue
        run_trial(ds_path,ds_label,opt_name,opt_obj)
print(f"\n✅ {MODEL_NAME} all trials done.")


[resume] found 4 completed trial(s) already in /content/drive/MyDrive/epilepsy_outputs/results.csv, skipping those:
    - Dataset_A | Adam
    - Dataset_A | AdamW
    - Dataset_A | Nadam
    - Dataset_B | Adam
[resume] skipping cnn_lstm | Dataset A | Adam (already done)
[resume] skipping cnn_lstm | Dataset A | Nadam (already done)
[resume] skipping cnn_lstm | Dataset A | AdamW (already done)
[resume] skipping cnn_lstm | Dataset B | Adam (already done)

  cnn_lstm | Dataset B | Nadam
    [resume] found checkpoint at epoch 27, continuing from there
Epoch 28/50
1798/1798 ━━━━━━━━━━━━━━━━━━━━ 67s 29ms/step - accuracy: 0.9993 - auc: 0.7387 - loss: 2.5687 - precision: 0.0000e+00 - recall: 0.0000e+00 - val_accuracy: 0.9994 - val_auc: 0.7966 - val_loss: 0.0218 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - learning_rate: 0.0010
Epoch 29/50
1798/1798 ━━━━━━━━━━━━━━━━━━━━ 49s 27ms/step - accuracy: 0.9994 - auc: 0.8376 - loss: 2.0131 - precision: 0.0000e+00 - recall: 0.0000e+00 - val_accu